In [ ]:
import numpy as np 
import numpy.ma as ma
import pandas as pd 
import os
import finlab
from finlab import data
import matplotlib.pyplot as plt

finlab_token = os.getenv("FINLAB_API_KEY")
finlab.login(finlab_token)

輸入成功!


In [2]:
VALID_DAYS = 270
MAX_STOCK_PER_DAY = 1
ROI_STOCK_PER_DAY = 1
PRED_LEN = 30
TAX_RATE = 0.003
FEE = 0.001425
THRESHOLD = 0.5
GROUPGAP = 30

In [3]:
from metrics.correlations import daily_correlation, stockly_correlation
from metrics.precisions import top_ratio_overlap_rate, top_ratio_overlap_rate_daily, precision_at_ratio, precision_daily_at_ratio, precision_baseline
from metrics.intervals import calculate_precision, extract_interval_matches, calculate_total_truth, group_intervals, get_prediction_interval_roi
from metrics.loss import mse_loss

In [4]:
df_close = data.get("etl:adj_close")

Your version is 1.2.20, please install a newer version.
Use "pip install finlab==1.3.0" to update the latest version.


In [5]:
result_root_dir = "../../results"

def positive(x):
    return x > 0.3

def analyze(pred_df, truth_df):
    pred_values = pred_df.values.flatten()
    truth_values = truth_df.values.flatten()
    
    corr = ma.corrcoef(ma.masked_invalid(pred_values), ma.masked_invalid(truth_values))[0][1]
    daily_corr = daily_correlation(pred_df, truth_df)
    stockly_corr = stockly_correlation(pred_df, truth_df)
    mse = mse_loss(pred_values, truth_values)
    
    pred_df = pred_df.fillna(-1)
    
    threshold = 0.5
    result = extract_interval_matches(pred_df, truth_df, threshold, group_gap=GROUPGAP)
    TP, total_positive, total_truth = calculate_precision(result)
    truth_intervals_dict, total_truth = calculate_total_truth(truth_df, threshold, GROUPGAP)
    
        
    precision_1 = precision_at_ratio(pred_df, truth_df, 0.01, positive)
    precision_daily_1 = precision_daily_at_ratio(pred_df, truth_df, 0.01, positive)
    precision_5 = precision_at_ratio(pred_df, truth_df, 0.05, positive)
    precision_daily_5 = precision_daily_at_ratio(pred_df, truth_df, 0.05, positive)
    precision_10 = precision_at_ratio(pred_df, truth_df, 0.1, positive)
    precision_daily_10 = precision_daily_at_ratio(pred_df, truth_df, 0.1, positive)
    baseline_precision = precision_baseline(truth_df, positive)
    
    return {
        "corr": corr,
        "daily_corr": daily_corr,
        "stockly_corr": stockly_corr,
        "TP": TP,
        "total_positive": total_positive,
        "total_truth": total_truth,
        "mse": mse,
        "precision@0.01": precision_1,
        "precision_daily@0.01": precision_daily_1,
        "precision@0.05": precision_5,
        "precision_daily@0.05": precision_daily_5,
        "precision@0.1": precision_10,
        "precision_daily@0.1": precision_daily_10,
        "baseline_precision": baseline_precision,
    }


result_dir = "../../results/AllTSE"
train_test_splits = ["2018_2020", "2019_2021", "2020_2022", "2021_2023"]

pred_file, truth_file = "pred_pct.csv", "truth_pct.csv"
flag = "test"

result_datas = []
result_datas_filtered = []
test_years = [2021, 2022, 2023, 2024]

# test_years = [2021]

for test_year in test_years:
    split = f"{test_year-3}_{test_year-1}"
    val_year = test_year - 1
    
    pred_file = os.path.join(result_dir, split, "test", "pred_pct.csv")
    truth_file = os.path.join(result_dir, split, "test", "truth_pct.csv")
    pred_train_val_file = os.path.join(result_dir, split, "train_val", "pred_pct.csv")
    truth_train_val_file = os.path.join(result_dir, split, "train_val", "truth_pct.csv")

    # 讀入當前 split 的預測 CSV
    def read_df(file_path):
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df.set_index('date', inplace=True)
        df = df.sort_index()
        return df
    
    test_pred_df = read_df(pred_file)
    test_truth_df = read_df(truth_file)
    
    train_val_pred_df = read_df(pred_train_val_file)
    train_val_truth_df = read_df(truth_train_val_file)
    
    val_pred_df = train_val_pred_df[train_val_pred_df.index.year == val_year]
    val_truth_df = train_val_truth_df[train_val_truth_df.index.year == val_year]
    
    val_corr = val_pred_df.corrwith(val_truth_df, axis=0).sort_values(ascending=False)
    top_corr_stocks = val_corr[:25].index.tolist()
    
    analysis = analyze(test_pred_df, test_truth_df)
    filtered_anaysis = analyze(test_pred_df[top_corr_stocks], test_truth_df[top_corr_stocks])
    
    result_datas.append(analysis)
    result_datas_filtered.append(filtered_anaysis)

result_df = pd.DataFrame(result_datas)
result_df_filtered = pd.DataFrame(result_datas_filtered)

FileNotFoundError: [Errno 2] No such file or directory: '../../results/AllTSE/2018_2020/test/pred_pct.csv'

In [ ]:
val_corr =  val_pred_df.corrwith(val_truth_df, axis=0).sort_values(ascending=False)
test_corr = test_pred_df.corrwith(test_truth_df, axis=0).sort_values(ascending=False)
print(val_corr.corr(test_corr).round(4))

0.0107


In [ ]:
result_df

,corr,daily_corr,stockly_corr,TP,total_positive,total_truth,mse,precision@0.01,precision_daily@0.01,precision@0.05,precision_daily@0.05,precision@0.1,precision_daily@0.1,baseline_precision
0,0.280540,0.223150,0.043475,29,76,122,0.020550,0.291031,0.181330,0.218237,0.158603,0.164330,0.137339,0.052103
1,0.225306,0.236310,-0.020717,3,25,50,0.012700,0.156882,0.165584,0.104676,0.112357,0.081481,0.083309,0.029129
2,0.223401,0.250012,-0.070529,1,27,114,0.016902,0.152791,0.155286,0.163697,0.161394,0.144298,0.150367,0.053500
3,0.184752,0.164552,0.027851,7,36,78,0.018846,0.162162,0.117403,0.137770,0.114264,0.107060,0.097606,0.044825


In [ ]:
result_df_filtered

,corr,daily_corr,stockly_corr,TP,total_positive,total_truth,mse,precision@0.01,precision_daily@0.01,precision@0.05,precision_daily@0.05,precision@0.1,precision_daily@0.1,baseline_precision
0,0.222552,0.182923,0.052398,0,0,2,0.008002,0.034483,0.115880,0.137457,0.115880,0.092784,0.083691,0.023348
1,0.300442,0.229759,0.032103,0,0,3,0.009234,0.052632,0.034632,0.187500,0.034632,0.178510,0.082251,0.039134
2,0.231827,0.180213,-0.033308,0,1,2,0.008656,0.517857,0.246696,0.197880,0.246696,0.104056,0.127753,0.025551
3,0.279921,0.308031,0.037346,0,1,4,0.013356,0.044444,0.254144,0.141593,0.254144,0.183628,0.190608,0.040884
